## Group No : 29
## Group Member Names:
| Sl. No. | Name (as appears in Canvas) | ID NO | Contribution (%) |
| :---: | :--- | :---: | :---: |
| **1** | CHETANA KOMMI | 2024da04169 | 100 |
| **2** | GOKULNATH R | 2024da04279 | 100 |
| **3** | RAJU DUBEY | 2024cm13025 | 100 |
| **4** | RAJESH KUMAR | 2024da04167 | 100 |

In [3]:
# Task 1: Import pandas and read the CSV file

import pandas as pd

# Read the dataset
df = pd.read_csv(r"E:\Learning\Python\IIT_Delhi_DiplomaProgram\DataScience_AI-ML_IIT-Delhi_CertificationProgram\email_spam.csv")

# Extract the 'title' column for further NLP tasks
titles = df['title']

# Display first few titles to verify
titles.head()

0                            ?? the secrets to SUCCESS
1                      ?? You Earned 500 GCLoot Points
2                           ?? Your GitHub launch code
3    [The Virtual Reward Center] Re: ** Clarifications
4    10-1 MLB Expert Inside, Plus Everything You Ne...
Name: title, dtype: object

## Task 2: Use TF-IDF Vectorization to create a vectorized document-term matrix. Justify the max_df and min_df parameters you used.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_df=0.9,
    min_df=5,
    stop_words='english'
)

# Create document-term matrix using the title column
tfidf_matrix = tfidf.fit_transform(titles)

# Shape of the TF-IDF matrix
tfidf_matrix.shape


(84, 3)

### Justification:
max_df = 0.9 : Removes words appearing in more than 90% of documents, as they are too common and carry little discriminative information.

min_df = 5 : Removes very rare words that appear in fewer than 5 documents, reducing noise and improving model robustness.

stop_words = 'english' : Eliminates common English words (e.g., the, is, and) that do not contribute to spam detection.

## Task 3: Build and display a dependency parser tree for the sentence at index 5 in the dataset.

In [23]:
# Install spaCy (run once if not installed)
!pip install spacy

In [24]:
# Install spaCy model (run once if not installed)
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------------------------- -------------- 8.1/12.8 MB 42.4 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 39.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import spacy
from spacy import displacy

# Load English NLP model
nlp = spacy.load("en_core_web_sm")

# Get the sentence at index 5 from the title column
sentence = titles.iloc[5]

print("Sentence at index 5:")
print(sentence)

# Apply dependency parsing
doc = nlp(sentence)

# Display dependency parse tree
displacy.render(doc, style="dep", jupyter=True)

Sentence at index 5:
AFE Model Casting Call 


### Explanation:
A dependency parser analyzes the grammatical structure of a sentence by identifying relationships between words.
Using spaCy’s pre-trained English model, the sentence at index 5 from the dataset was parsed to visualize syntactic dependencies such as subject, object, and modifiers.
The dependency tree helps in understanding sentence structure, which is useful for downstream NLP tasks like information extraction and intent detection.

## Task 4: Using Scikit-Learn, create an instance of LDA with 20 components (random_state=42).

In [6]:
from sklearn.decomposition import LatentDirichletAllocation

# Create LDA model with 20 topics
lda_model = LatentDirichletAllocation(
    n_components=20,
    random_state=42
)

# Fit LDA model on TF-IDF document-term matrix
lda_model.fit(tfidf_matrix)

lda_model


,n_components,20
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


### Explanation:
Latent Dirichlet Allocation (LDA) is an unsupervised topic modeling technique used to identify latent topics in a corpus.
Using Scikit-Learn, an LDA model with 20 components was initialized and trained on the TF-IDF document-term matrix.
The random_state parameter ensures reproducible results.

## Task 5: Print out the top 15 most common words for each of the 20 topics.

In [7]:
import numpy as np

# Get feature (word) names from TF-IDF vectorizer
feature_names = tfidf.get_feature_names_out()

# Number of top words to display per topic
n_top_words = 15

# Print top words for each topic
for topic_idx, topic in enumerate(lda_model.components_):
    top_features_idx = topic.argsort()[:-n_top_words - 1:-1]
    top_words = [feature_names[i] for i in top_features_idx]
    
    print(f"Topic {topic_idx + 1}:")
    print(", ".join(top_words))
    print("-" * 50)


Topic 1:
new, account, job
--------------------------------------------------
Topic 2:
job, new, account
--------------------------------------------------
Topic 3:
new, account, job
--------------------------------------------------
Topic 4:
new, account, job
--------------------------------------------------
Topic 5:
new, account, job
--------------------------------------------------
Topic 6:
new, account, job
--------------------------------------------------
Topic 7:
new, account, job
--------------------------------------------------
Topic 8:
new, account, job
--------------------------------------------------
Topic 9:
new, account, job
--------------------------------------------------
Topic 10:
new, account, job
--------------------------------------------------
Topic 11:
new, account, job
--------------------------------------------------
Topic 12:
new, account, job
--------------------------------------------------
Topic 13:
new, account, job
---------------------------------

### Explanation: 
The most representative words for each topic were extracted by sorting the LDA topic-word distribution and selecting the top 15 terms per topic.

## Task 6: Add a new column to the original Data Frame that labels each question into one of the 20 topic categories.

In [8]:
# Transform documents to topic distribution
doc_topic_dist = lda_model.transform(tfidf_matrix)

# Assign the dominant topic to each document
df['topic_label'] = doc_topic_dist.argmax(axis=1) + 1  # +1 for human-readable topic numbers

# Display first few rows to verify
df[['title', 'topic_label']].head()


,title,topic_label
0,?? the secrets to SUCCESS,1
1,?? You Earned 500 GCLoot Points,1
2,?? Your GitHub launch code,1
3,[The Virtual Reward Center] Re: ** Clarifications,1
4,"10-1 MLB Expert Inside, Plus Everything You Ne...",1


### Explanation:
Each document was assigned to the topic with the highest probability from the LDA topic distribution. The resulting topic label was added as a new column to the original dataset.

##                                                End of Assignment 